# Traffic Light Decision Framework & Actionable Thresholds

## 1. Operational Risk Thresholds
The continuous $0\text{--}100\%$ risk score generated by the XGBoost engine is mapped to three strict operational tiers to guide automated alerts and field actions:

*   🟢 **GREEN (0 – 39% Risk) — Low Risk / Normal Operations:** Asset operating within safe thermal/electrical thresholds. *Action:* Continue standard SCADA monitoring.
*   🟡 **AMBER (40 – 74% Risk) — Moderate Risk / Elevated Stress:** Asset under elevated weather or load stress. *Action:* Queue for priority field inspection and stage replacement inventory.
*   🔴 **RED (75 – 100% Risk) — High Risk / Critical Failure Imminent:** Asset exceeding safe operating parameters under severe weather conditions. *Action:* Trigger immediate control room alert, initiate hazard-aware field crew dispatch, or execute automated load-shedding protocols.

---
## 2. Model Explainability (SHAP Pseudocode & Practical Meaning):
   Decomposes individual asset risk predictions into feature-level contributions so operators understand *why* a specific asset scored high. 
   
   *What this means with an example:* If Substation 102 receives an alarming risk score of **87% (RED)**, the explainability module breaks down the score to show operators that it didn't just happen randomly—for instance, it might indicate:
   - $+35\%$ risk contributed by high wind speeds during a storm,
   - $+25\%$ risk contributed by a pending overdue maintenance log, and
   - $+15\%$ risk from heavy electrical power load. 
   This transparency builds operator trust and ensures field crews are dispatched with the exact context of the failure hazard.

## 3. Control Room Dashboard

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# 1. Load the risk scores dataset
df = pd.read_csv('../data/df_featured_risk_scores.csv')

# 2. Compute High-Level Metrics
total_assets = len(df)
high_risk_count = len(df[df['risk_score_0_100'] >= 75])
avg_grid_risk = df['risk_score_0_100'].mean()

print(f"--- CONTROL ROOM SUMMARY ---")
print(f"Total Assets Monitored : {total_assets}")
print(f"High Risk Count (RED)  : {high_risk_count}")
print(f"Average Grid Risk Score: {avg_grid_risk:.2f}%")

# 3. Render Score Distribution Histogram with Traffic Light Zones
fig = px.histogram(
    df, 
    x='risk_score_0_100', 
    nbins=20, 
    title='Control Room: Asset Risk Score Distribution',
    labels={'risk_score_0_100': 'Risk Score (%)', 'count': 'Number of Assets'},
    color_discrete_sequence=['#1f77b4']
)

# Add visual thresholds for Green, Amber, Red zones
fig.add_vrect(x0=0, x1=39, fillcolor="green", opacity=0.15, layer="below", line_width=0, annotation_text="GREEN (Normal)")
fig.add_vrect(x0=40, x1=74, fillcolor="orange", opacity=0.15, layer="below", line_width=0, annotation_text="AMBER (Elevated)")
fig.add_vrect(x0=75, x1=100, fillcolor="red", opacity=0.15, layer="below", line_width=0, annotation_text="RED (Critical)")

fig.update_layout(bargap=0.1, template='plotly_white')
fig.show()

## 4. Geospatial Risk Map

In [ ]:
import folium
from folium.plugins import MarkerCluster

# 1. Initialize map centered around average asset coordinates
df[['latitude', 'longitude']] = df['Fault Location (Latitude, Longitude)'].str.strip('()').str.split(', ', expand=True).astype(float)
map_center = [df['latitude'].mean(), df['longitude'].mean()]
risk_map = folium.Map(location=map_center, zoom_start=11)

# Marker cluster for performance
marker_cluster = MarkerCluster().add_to(risk_map)

# 2. Function to assign traffic light colors based on risk score
def get_traffic_light_color(score):
    if score >= 75:
        return 'red'
    elif score >= 40:
        return 'orange'
    else:
        return 'green'

# 3. Plot each asset onto the map
for _, row in df.iterrows():
    score = row['risk_score_0_100']
    color = get_traffic_light_color(score)
    
    popup_html = f"""
    <b>Asset ID:</b> {row.get('asset_id', 'N/A')}<br>
    <b>Type:</b> {row.get('asset_type', 'Substation/Station')}<br>
    <b>Risk Score:</b> {score:.1f}%<br>
    <b>Status Tier:</b> {color.upper()}
    """
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=folium.Popup(popup_html, max_width=300)
    ).add_to(marker_cluster)

# Display the interactive map
risk_map